# Predicting Negative Customer Reviews in Brazilian E-Commerce

This notebook is my compact end-to-end walkthrough of the project. I compare negative-review prediction at order placement with prediction after delivery, add an unsupervised segmentation layer, and connect the analytical results to a Power BI deployment workflow.

**Target:** `review_bad = 1` for review scores 1–2 and `0` for scores 3–5.

**Final sample:** 95,824 delivered orders; negative-review rate approximately 12.8%.

## 1. Analytical design

I use the same held-out orders for the placement and delivery models. This keeps the comparison focused on information availability rather than differences in the test sample. Delivery outcomes are excluded from the placement model to avoid look-ahead leakage.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

comparison = pd.read_csv('../outputs/model_metrics/placement_vs_delivery.csv')
comparison


## 2. Order segmentation

I use K-Means to create four descriptive order personas from order value, freight, product weight, delivery duration, delivery performance relative to promise, installments and number of items. Review outcome is not used to fit the clusters.

The silhouette score is modest (~0.204), so I treat the personas as useful descriptive profiles rather than sharply separated natural groups.

In [ ]:
clusters = pd.read_csv('../outputs/interpretation/cluster_profile.csv')
clusters[['cluster','persona','share','bad_review_rate','avg_delivery_time_days','avg_n_items']]


The **Long Wait** and **Multi-Item Basket** groups have the highest negative-review rates. This shows that lateness is important, but dissatisfaction is not explained by lateness alone.

## 3. Placement versus delivery

The placement model uses only information available when the order is created. The delivery model adds observed fulfillment information. The earlier score creates more time to intervene; the later score provides a stronger risk signal.

In [ ]:
comparison.set_index('stage')[['precision','recall','f1','roc_auc']].round(3)


In [ ]:
ax = comparison.set_index('stage')[['f1','roc_auc']].plot(kind='bar', figsize=(7,4))
ax.set_ylim(0, 1)
ax.set_ylabel('Score')
ax.set_title('Predictive performance improves after fulfillment information is observed')
plt.xticks(rotation=0)
plt.tight_layout()


In the current rebuild, F1 increases from about **0.302** at placement to **0.486** at delivery, while ROC-AUC increases from about **0.658** to **0.774**. The placement model retains higher recall, while the delivery model is substantially more precise.

## 4. Final-model interpretation

CatBoost is my final delivery-stage model. I use feature importance as a predictive diagnostic rather than a causal explanation.

In [ ]:
importance = pd.read_csv('../outputs/interpretation/catboost_feature_importance.csv')
importance.head(10)


In [ ]:
top = importance.head(10).sort_values('importance')
plt.figure(figsize=(8,5))
plt.barh(top['feature'], top['importance'])
plt.xlabel('Feature importance')
plt.title('At-delivery CatBoost: leading predictive features')
plt.tight_layout()


The strongest features are `late_days`, `delivery_vs_estimate_days`, `n_items` and `delivery_time_days`. Fulfillment problems dominate the risk signal, while order complexity contributes additional information.

## 5. Business interpretation and deployment

I use the placement score for lower-cost monitoring and early communication, then use the stronger delivery-stage score for more selective service-recovery prioritization.

Notebook 06 joins the order-level analytical fields, personas and saved model probabilities into one curated CSV for Power BI. The dashboard is therefore a deployment layer on top of the same Python workflow, not a separate analysis.

The project does not estimate the causal effect of an intervention. A production test would need to measure whether acting on model flags changes review outcomes or customer economics.